# Optimize enhancer specificity with Malinois

Design 200-nt enhancers by maximizing K562 activity minus the higher of the HepG2 and SK-N-SH predictions. Malinois scores the 200-nt insert with the two fixed 200-nt MPRA flanks.

Fig. 2 uses base seed 42, FP32 and fixed per-optimizer prediction batches. Run this notebook in a fresh GPU kernel with MEME Suite on PATH. Keep the motif-discovery helper in the same directory.

In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"
import os
from pathlib import Path

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import numpy as np
import torch
from bioseq_models import load_model
from bioseq_models.adapters.malinois import LEFT_CONTEXT, RIGHT_CONTEXT
from MotifSift import call_motifs, random_seed_sequences, select_candidates
from fig2_motif_catalog import combine_seqlet_motifs
from sequence_optimization import MultiTaskObjective, SequencePredictor, onehot
from sequence_optimization.api import as_score_batch
from sequence_optimization.suite import generate_designs

torch.set_num_threads(1)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True)

CELLS = ("K562", "HepG2", "SKNSH")

class ContextPredictor(SequencePredictor):
    def prepare_input(self, values):
        left = self.left.to(values).expand(len(values), -1, -1)
        right = self.right.to(values).expand(len(values), -1, -1)
        return torch.cat((left, values, right), dim=1).transpose(1, 2)

class SpecificityScorer:
    supports_soft_scoring = True

    def __init__(self, raw, target):
        self.raw = raw
        self.target = CELLS.index(target)
        self.off = [i for i in range(3) if i != self.target]
        self.task_names = (*CELLS, *(f'{target}_minus_{CELLS[i]}' for i in self.off))
        self.device = 'cuda'
        self.soft = ContextPredictor(raw.model, device='cuda', tasks=CELLS, batch_size=1024)
        self.soft.left = torch.from_numpy(onehot([LEFT_CONTEXT], channels_first=False))
        self.soft.right = torch.from_numpy(onehot([RIGHT_CONTEXT], channels_first=False))

    def score(self, sequences):
        x = as_score_batch(self.raw.score(sequences)).scores
        return np.column_stack([x, *(x[:, self.target] - x[:, i] for i in self.off)])

    def score_soft(self, probabilities):
        x, _ = self.soft.score_soft(probabilities)
        return torch.cat([x, *(x[:, self.target:self.target+1] - x[:, i:i+1]
                               for i in self.off)], dim=1), None

model = SpecificityScorer(load_model("malinois", device="cuda", batch_size=1024), "K562")
objective = MultiTaskObjective(high=[3, 4], aggregation="worst")
run = Path(os.environ.get("MOTIFSIFT_FIG2_RUN", "outputs/malinois_k562"))
run.mkdir(parents=True, exist_ok=True)


## 1. Generate candidates

Run the eleven optimizers in a fixed order and keep 100 candidates per optimizer at 90% gain.

In [2]:
seeds = random_seed_sequences(200, 100, seed=42)
designs = generate_designs(
    model, length=200, initial=seeds, objective=objective,
    seed=42, parallel=False, output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])

,size,median,max
algorithm,,,
adalead,100,28.846767,30.252395
basesearch,100,23.288496,25.481710
beam_search,100,27.427982,30.767609
cross_entropy_method,100,28.943088,29.598214
den,100,20.580245,22.195775
fast_seqprop,100,26.567592,28.315869
genetic_algorithm,100,33.412852,34.887352
ism_greedy,100,28.653624,31.538803
ledidi,100,25.115909,27.679376


## 2. Discover motifs

Discover PWMs from all seqlets, then repeat discovery with one representative for each seqlet sequence or reverse complement. Apply the same FIMO support and spatial redundancy filters to the combined catalog using the complete candidate pool.

In [3]:
call_motifs(designs, model, objective=objective, output_dir=run / "motifs_first_pass")
motifs = combine_seqlet_motifs(run / "motifs_first_pass", run / "motifs")
motifs.support.query("retained == True")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]

,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-VWGATAAGG,3064,1052,5079.0
1,2-AMCAKMTGKT,517,464,1547.0
2,3-RTGASTCAY,262,213,955.0
4,5-AGCTATTTTTGGAA,16,16,149.0
6,9-ACCAGAC,187,183,282.0
7,10-CACACCCAH,16,16,571.0
22,balanced-4-TTTCCAGGAA,46,40,697.0
24,balanced-6-TAGGAACAGA,14,13,37.0
25,balanced-8-CTTCCCGCCAA,11,11,57.0
27,balanced-12-CACACCC,67,67,657.0


## 3. Select a panel

Select ten sequences above the 20th percentile of unique candidate scores.

In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
panel = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
panel[["selection_rank", "sequence", "score", "present_motifs"]]

,selection_rank,sequence,score,present_motifs
0,1,GTTATCAAACCGCTGTGTCTGGTTCGTACAGATAACGCTGACATTT...,34.887352,1-VWGATAAGG;2-AMCAKMTGKT;3-RTGASTCAY;9-ACCAGAC...
567,2,TCTAAGAATTACGCAACGCAATGTGTGAAATGAGAGATAAGGGCGC...,26.323567,1-VWGATAAGG
400,3,TCTTGCGCGAGGGTGCCTTATCTGTTCCTTGCAGCACTGTGGGGTT...,28.136970,1-VWGATAAGG;balanced-4-TTTCCAGGAA;balanced-6-T...
216,4,TATCTTCAGGGAACATCTGGTGAACACAGATAAGGCGTTCTCCAGT...,28.986864,1-VWGATAAGG;2-AMCAKMTGKT;9-ACCAGAC;balanced-13...
814,5,GACTAGCCACACCTTTGGGGTGTGAATCATCCCGCCAAACCGTTAT...,23.522303,1-VWGATAAGG;3-RTGASTCAY;10-CACACCCAH;balanced-...
458,6,TTATCACGCCAGAGGCCTATGTCCGTTCAGATAAAGCGGTTACTGG...,27.408631,1-VWGATAAGG;3-RTGASTCAY;5-AGCTATTTTTGGAA;balan...
714,7,TTGATAACCGTTTTTGGCCACTCCTGCCTGCTGTTTCCTGATAACA...,24.342896,1-VWGATAAGG;2-AMCAKMTGKT;balanced-12-CACACCC
743,8,GCTGACTCAGGGCGTGGTATCGCTACTTCTGCATATGTGAGATAAG...,24.067966,1-VWGATAAGG;2-AMCAKMTGKT;3-RTGASTCAY;balanced-...
764,9,TTATCACGCCAGAAGCATATGTCCGTTTAGATAAAGCGGTTACTGG...,23.861217,1-VWGATAAGG;2-AMCAKMTGKT;5-AGCTATTTTTGGAA;bala...
854,10,CCTAGAGATAAGACCTATAGGGTGTGACCTAAAATAGCTTTCTCAG...,23.010358,1-VWGATAAGG;2-AMCAKMTGKT;5-AGCTATTTTTGGAA;10-C...
